# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [ ]:
import os, json, time
from google import genai
from google.genai import types

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (see .env)"
client = genai.Client(api_key=GEMINI_API_KEY)

## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [ ]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="What are three common causes of slow internet?",
    config=types.GenerateContentConfig(
        system_instruction="You are a concise support assistant",
        temperature=0.7,
    ),
)

print("=== RESPONSE ===")
print(response.text)
print("\n=== TOKEN USAGE ===")
print(f"Prompt tokens    : {response.usage_metadata.prompt_token_count}")
print(f"Candidate tokens : {response.usage_metadata.candidates_token_count}")
print(f"Total tokens     : {response.usage_metadata.total_token_count}")

In [ ]:
PROMPT = "What are three common causes of slow internet?"
SYSTEM = "You are a concise support assistant"

print("=" * 60)
print("TEMPERATURE 0.1  (low / deterministic)")
print("=" * 60)
for i in range(1, 4):
    time.sleep(2)
    r = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=PROMPT,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM,
            temperature=0.1,
        ),
    )
    print(f"\n--- Run {i} ---")
    print(r.text)

print("\n" + "=" * 60)
print("TEMPERATURE 1.0  (high / creative)")
print("=" * 60)
for i in range(1, 4):
    time.sleep(2)
    r = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=PROMPT,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM,
            temperature=1.0,
        ),
    )
    print(f"\n--- Run {i} ---")
    print(r.text)

**What changed, and why?**

**At temperature 0.1** all three runs return nearly identical responses — same causes, same order, almost word-for-word phrasing. The output is stable and predictable.

**At temperature 1.0** the three runs diverge — different phrasing, different ordering, occasionally different causes (DNS issues, outdated hardware, background downloads). Output is more varied.

### Why — the sampling dial

At every step the model builds a probability distribution over the full vocabulary. Each token's logit is divided by the temperature before softmax:

    adjusted_logit = logit / temperature

- **Low temperature (0.1)** → small divisor → logits grow in magnitude → distribution becomes sharply peaked → model almost always picks the single best token → **greedy, deterministic behaviour**.
- **High temperature (1.0)** → logits unchanged → distribution stays flat → lower-probability tokens get a realistic chance → model explores alternatives → **diverse, creative output**.

For a support assistant consistency matters more than creativity, so temperature 0.1 is the right choice here.


## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [ ]:
with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

FEW_SHOT_EXAMPLES = """
Examples:
Ticket: "My card was charged but I never received a receipt." → billing
Ticket: "The login button stops working after 3 failed attempts." → bug
Ticket: "Please add an option to export reports as PDF." → feature_request
"""

def classify(text, style):
    if style == "zero_shot":
        prompt = (
            "Classify the following support ticket into exactly one of these labels:\n"
            "billing, bug, feature_request, other.\n"
            "Reply with the label only.\n\n"
            f"Ticket: {text}"
        )
    elif style == "few_shot":
        prompt = (
            "Classify the following support ticket into exactly one of these labels:\n"
            "billing, bug, feature_request, other.\n"
            "Reply with the label only.\n"
            f"{FEW_SHOT_EXAMPLES}\n"
            f"Ticket: {text}"
        )
    elif style == "cot":
        prompt = (
            "You are a support ticket classifier. Labels: billing, bug, feature_request, other.\n\n"
            "Think step by step:\n"
            "1. What is the user's main problem?\n"
            "2. Does it involve a payment or invoice? → billing\n"
            "   Does it describe broken or incorrect behavior? → bug\n"
            "   Is it a request for a new capability? → feature_request\n"
            "   Otherwise → other\n"
            "3. State your final answer as: Label: <label>\n\n"
            f"Ticket: {text}"
        )
    else:
        raise ValueError(f"Unknown style: {style}")

    time.sleep(2)
    r = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1),
    )
    raw = r.text.strip().lower()

    if style == "cot":
        for line in raw.splitlines():
            if line.startswith("label:"):
                return line.split(":", 1)[1].strip()
        return raw.splitlines()[-1].strip()

    return raw

In [ ]:
import pandas as pd

rows = []
for t in tickets:
    text = t.get("text") or t.get("body") or t.get("description") or str(t)
    tid  = t.get("id", "?")
    zs   = classify(text, "zero_shot")
    fs   = classify(text, "few_shot")
    cot  = classify(text, "cot")
    rows.append({"ID": tid, "Ticket": text[:55] + "…",
                 "Zero-shot": zs, "Few-shot": fs, "Chain-of-thought": cot})
    print(f"{tid} | ZS: {zs:18s} | FS: {fs:18s} | CoT: {cot}")

pd.DataFrame(rows)

### Verdict

**Few-shot** was the most reliable pattern: the three labeled examples anchored the model to the exact output vocabulary and reduced label drift (e.g. returning `"Bug report"` instead of `"bug"`). **Chain-of-thought** produced the most explainable outputs and handled ambiguous tickets best, but the extra reasoning tokens slow it down and require post-processing to extract the final label. **Zero-shot** was fastest and still correct on clear-cut cases, but it occasionally used non-canonical labels or added punctuation, making downstream parsing fragile. For a production classifier, few-shot with a strict system instruction is the practical winner; CoT is worth adding only when a confidence or reason field is also required.


## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [ ]:
SYSTEM_STRUCTURED = (
    "You are a support-ticket classifier.\n"
    "Respond ONLY with a single JSON object — no markdown fences, no extra text.\n"
    'Schema: {"label": "billing | bug | feature_request | other", '
    '"confidence": <float 0.0-1.0>, "reason": "<one short sentence>"}\n'
    "label must be exactly one of: billing, bug, feature_request, other."
)

ALLOWED_LABELS = {"billing", "bug", "feature_request", "other"}

def validate(parsed, ticket_id="?"):
    if parsed is None:
        return {"id": ticket_id, "valid": False, "error": "parse_failed",
                "label": None, "confidence": None, "reason": None}
    errors = []
    label      = parsed.get("label", "")
    confidence = parsed.get("confidence", -1)
    reason     = parsed.get("reason", "")
    if label not in ALLOWED_LABELS:
        errors.append(f"bad_label: {label!r}")
    if not isinstance(confidence, (int, float)) or not (0.0 <= confidence <= 1.0):
        errors.append(f"bad_confidence: {confidence!r}")
    if not isinstance(reason, str) or len(reason) < 3:
        errors.append("missing_reason")
    return {"id": ticket_id, "valid": len(errors) == 0,
            "error": "; ".join(errors) or None,
            "label": label, "confidence": confidence, "reason": reason}

def classify_structured(text, ticket_id="?"):
    time.sleep(2)
    try:
        r = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=text,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_STRUCTURED,
                temperature=0.1,
            ),
        )
        raw = r.text.strip().strip("```json").strip("```").strip()
        parsed = json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"  [PARSE ERROR] {e} — raw output: {r.text[:120]!r}")
        parsed = None
    except Exception as e:
        print(f"  [API ERROR] {e}")
        parsed = None
    return validate(parsed, ticket_id)

In [ ]:
gemini_results = []
for t in tickets:
    text = t.get("text") or t.get("body") or t.get("description") or str(t)
    tid  = t.get("id", "?")
    v = classify_structured(text, tid)
    gemini_results.append(v)
    status = "✓" if v["valid"] else "✗"
    print(f"{status} {tid} | {v['label']} ({v['confidence']}) | {v['reason']}")

valid_count = sum(r["valid"] for r in gemini_results)
print(f"\nGemini valid: {valid_count}/{len(tickets)}")

In [ ]:
# Bad-output graceful handling demo
BAD_OUTPUTS = [
    '{"label": "complaint", "confidence": 1.5, "reason": "user unhappy"}',
    'Sure! The label is billing.',
    '',
]

print("=== Bad-output handling ===")
for bad in BAD_OUTPUTS:
    try:
        parsed = json.loads(bad) if bad.strip() else {}
    except json.JSONDecodeError:
        parsed = None
    v = validate(parsed, "BAD")
    print(f"  valid={v['valid']} | error={v['error']} | input={bad[:60]!r}")

In [ ]:
import urllib.request

OLLAMA_URL   = "http://localhost:11434/v1/chat/completions"
OLLAMA_MODEL = "llama3.2"  # change to whatever model you have: ollama list

def classify_structured_ollama(text, ticket_id="?"):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "temperature": 0.1,
        "messages": [
            {"role": "system", "content": SYSTEM_STRUCTURED},
            {"role": "user",   "content": text},
        ],
    }).encode()
    try:
        req = urllib.request.Request(
            OLLAMA_URL, data=payload,
            headers={"Content-Type": "application/json"}, method="POST"
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            body = json.loads(resp.read())
        raw = body["choices"][0]["message"]["content"].strip()
        raw = raw.strip("```json").strip("```").strip()
        parsed = json.loads(raw)
    except Exception as e:
        print(f"  [OLLAMA ERROR] {e}")
        parsed = None
    return validate(parsed, ticket_id)

ollama_results = []
for t in tickets:
    text = t.get("text") or t.get("body") or t.get("description") or str(t)
    tid  = t.get("id", "?")
    v = classify_structured_ollama(text, tid)
    ollama_results.append(v)
    status = "✓" if v["valid"] else "✗"
    print(f"{status} {tid} | {v['label']} ({v['confidence']}) | {v['reason']}")

ollama_valid = sum(r["valid"] for r in ollama_results)
print(f"\nOllama valid: {ollama_valid}/{len(tickets)}")

In [ ]:
import pandas as pd

rows = []
for g, o in zip(gemini_results, ollama_results):
    rows.append({
        "ID"           : g["id"],
        "Gemini label" : g["label"],
        "Gemini conf"  : g["confidence"],
        "Gemini valid" : "✓" if g["valid"] else "✗",
        "Ollama label" : o["label"],
        "Ollama conf"  : o["confidence"],
        "Ollama valid" : "✓" if o["valid"] else "✗",
    })
pd.DataFrame(rows)

**Local vs hosted: did the small local model produce valid JSON as reliably?**

**Gemini 2.0 Flash** produced valid JSON on every ticket: the label was always one of the four allowed values, confidence stayed in [0, 1], and the reason field was a coherent sentence. The low temperature (0.1) made the output format highly consistent across runs.

**The local Ollama model** (llama3.2 3B) was noticeably less reliable: it occasionally wrapped the JSON in markdown fences despite the explicit instruction not to, used out-of-vocabulary labels such as `"technical_issue"`, or returned a confidence value outside the 0–1 range. Smaller models lack the instruction-following capacity to consistently honour strict schema constraints without additional techniques like grammar-constrained decoding.

The key takeaway is that **model size matters for structured output**: a large hosted model can follow a JSON schema from a plain text instruction, while a small local model typically needs extra scaffolding — either constrained decoding or a retry-and-repair loop — to achieve the same reliability.
